# Full pipeline

Neural Network-Driven Signal Generation for Systematic Portfolio Construction (MScFE 690 Capstone).

This notebook holds no logic of its own. Each step calls a module in `src/` and shows the result. Settings are in `config.yaml`.

In [ ]:
import os, sys

if "google.colab" in sys.modules and not os.path.exists("src"):
    !git clone https://github.com/OWNER/REPO.git repo
    %cd repo
    !pip install -q -r requirements.txt

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src import data, features, splits

cfg = data.load_config(ROOT / "config.yaml")
cfg["run"]

## 1. Data

Yahoo prices and FRED macro series, cached in `data/raw`. In quick mode the prices are generated instead.

In [ ]:
if cfg["run"]["quick_mode"]:
    prices, macro = data.make_synthetic(cfg, n_days=2600)
else:
    prices, macro = data.load_data(cfg)

print(f"{len(prices)} trading days, {prices.index.min().date()} to {prices.index.max().date()}")
macro.dropna().tail()

## 2. Features and label

31 indicators per ETF. The label for a decision at the close of day t is the return from the close of t+1 to the close of t+6.

In [ ]:
dataset = features.build_dataset(prices, macro, cfg)
feature_cols = features.feature_columns(dataset)

print(f"{len(dataset):,} rows, {len(feature_cols)} features")
dataset[feature_cols].describe().T[["mean", "std", "min", "max"]].round(3)

In [ ]:
balance = dataset.groupby("ticker")["label"].mean().sort_values(ascending=False)

ax = balance.plot(kind="bar", color="steelblue", figsize=(11, 4), ylim=(0.4, 0.65))
ax.axhline(0.5, color="red", linestyle="--", linewidth=1, label="50/50")
ax.set_title("Share of positive 5-day labels by ETF")
ax.set_ylabel("Positive share")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Walk-forward splits

Training expands by one year at each step, and the last 12 months of each training window are the validation block. An 11-day gap (1 execution lag + 5 horizon + 5 embargo) is left before validation and before the test year, so each test year is complete.

In [ ]:
folds = splits.walk_forward_splits(prices.index, cfg)

pd.DataFrame(
    [
        {
            "test_year": s.test_year,
            "train_start": s.train.min().date(),
            "train_end": s.train.max().date(),
            "validation_start": s.validation.min().date(),
            "validation_end": s.validation.max().date(),
            "test_start": s.test.min().date(),
            "test_end": s.test.max().date(),
        }
        for s in folds
    ]
).set_index("test_year")

## 4. Models, 5. Signals, 6. Portfolio, 7. Evaluation and statistics

To be added as `src/models.py`, `src/signals.py`, `src/portfolio.py`, `src/evaluate.py` and `src/stats.py` are implemented.